# E3-F2B — Réentraînement contrôlé LSTM / TCN multidevices (200 époques)

**Objectif :** tester l'effet d'un budget maximal de **200 époques**, patience **25**, sur les 24 entraînements finaux E3-E3 (4 horizons × 2 modèles × 3 graines), sans modifier les résultats historiques.

**Important :** nouveaux entraînements **depuis l'initialisation**, et non reprise exacte des checkpoints E3-E3. Architectures déjà sélectionnées dans E3-E3, **sans nouvelle recherche sur le test**. Toutes les prédictions sont calculées sur les **13 643 mêmes clés de test**.

**Comparaison :** les performances E3-E3 à 80 époques et E3-F2B à 200 époques sont des exécutions contrôlées du même protocole, mais avec des trajectoires d'optimisation potentiellement différentes. La sélection du meilleur checkpoint repose uniquement sur la validation.


In [ ]:
from pathlib import Path
import json, random, time, platform
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT=Path.cwd().resolve()
REP=ROOT/'outputs/E3_aligned_representations'
COMMON=ROOT/'outputs/E3_common_evaluation'
OUT=ROOT/'outputs/E3_F2B_LSTM_TCN_200epochs'
PREVIOUS=ROOT/'outputs/E3_E3_multidevice_LSTM_TCN_105'
HORIZONS=(15,30,60,120)
SPLITS=('train','validation','test')
SEEDS=(42,123,2026)
EXPECTED={'train':64393,'validation':13727,'test':13643}
KEYS=['uuid','timestamp','split']
BATCH_SIZE=256
MAX_EPOCHS=200
PATIENCE=25
LR=1e-3
WEIGHT_DECAY=1e-5
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

assert REP.is_dir() and COMMON.is_dir() and PREVIOUS.is_dir(), 'E3_aligned_representations / E3_common_evaluation absent.'
OUT.mkdir(parents=True,exist_ok=True)
common=pd.read_csv(COMMON/'01_common_keys_all_horizons.csv')
common['timestamp']=pd.to_datetime(common['timestamp'],utc=True)
assert common.groupby('split').size().to_dict()==EXPECTED
assert not common.duplicated(KEYS).any()
print('Project:',ROOT,'Device:',DEVICE,'PyTorch:',torch.__version__)

# Fixed architectures selected by validation in E3-E3, no new architecture selection.
old_configs=pd.read_csv(PREVIOUS/'05_selected_configs_and_training.csv')
assert len(old_configs)==24
assert not old_configs.duplicated(['horizon_min','model','seed']).any()
assert set(old_configs.seed)==set(SEEDS)
selected=old_configs.groupby(['horizon_min','model'])['configuration'].agg(lambda s: s.unique().tolist())
assert all(len(names)==1 for names in selected)
print('Architectures fixed from E3-E3:',selected.to_dict())


In [ ]:
def load_horizon(h):
    folder=REP/f'h{h:03d}'
    node_table=pd.read_csv(folder/'01_graph_node_index.csv').sort_values('node_index')
    devices=node_table['uuid'].astype(str).tolist()
    assert node_table['node_index'].tolist()==list(range(7)) and len(set(devices))==7
    data={}
    for split in SPLITS:
        Xg=np.load(folder/f'X_graph_{split}_scaled.npy',mmap_mode='r')
        Xu=np.load(folder/f'X_graph_{split}_unscaled.npy',mmap_mode='r')
        Y=np.load(folder/f'y_graph_{split}.npy').astype(np.float32)
        meta=pd.read_csv(folder/f'metadata_{split}.csv')
        for col in ('timestamp','target_timestamp'):
            meta[col]=pd.to_datetime(meta[col],utc=True)
        idx=np.load(COMMON/f'indices_h{h:03d}_{split}.npy').astype(np.int64)
        S=len(Y);N=S*7
        assert Xg.shape==Xu.shape==(S,5,7,3),f'Unexpected shape: {Xg.shape}'
        assert Y.shape==(S,7) and len(meta)==N
        assert meta['uuid'].astype(str).to_numpy().reshape(S,7).tolist()==[devices]*S
        assert (meta.target_timestamp-meta.timestamp).eq(pd.Timedelta(minutes=h)).all()
        assert np.allclose(meta['target'].to_numpy(),Y.reshape(-1),rtol=0,atol=1e-5)
        assert len(idx)==EXPECTED[split] and len(set(idx.tolist()))==len(idx)
        assert idx.min()>=0 and idx.max()<N
        got=meta.iloc[idx][KEYS].sort_values(KEYS).reset_index(drop=True)
        expected=common.loc[common.split.eq(split),KEYS].sort_values(KEYS).reset_index(drop=True)
        assert got.equals(expected),f'Common keys mismatch h={h}, {split}'
        # Same 105 values as E3-E2, reorganized as 5 timesteps × 21 channels.
        X=np.asarray(Xg).reshape(S,5,21).astype(np.float32)
        persistence=np.asarray(Xu[:,-1,:,0],dtype=np.float64).reshape(-1)
        assert np.isfinite(X).all() and np.isfinite(Y).all()
        assert np.array_equal(X.reshape(S,105),np.asarray(Xg).reshape(S,105))
        data[split]=dict(X=X,Y=Y,meta=meta,idx=idx,persistence=persistence)
        print(f'h={h:3d}, {split:10s}: {S} snapshots; {N} nodes; common={len(idx)}')
    return data,devices

def metrics(y,p):
    y=np.asarray(y,dtype=np.float64).reshape(-1)
    p=np.asarray(p,dtype=np.float64).reshape(-1)
    e=p-y;den=np.sum((y-y.mean())**2)
    return {'n':int(len(y)),'RMSE':float(np.sqrt(np.mean(e*e))),
            'MAE':float(np.mean(np.abs(e))),
            'R2':float(1-np.sum(e*e)/den) if den>0 else np.nan,
            'Bias':float(np.mean(e))}


In [ ]:
class MultiLSTM(nn.Module):
    def __init__(self,hidden=32):
        super().__init__()
        self.lstm=nn.LSTM(input_size=21,hidden_size=hidden,batch_first=True)
        self.head=nn.Linear(hidden,7)
    def forward(self,x):
        z,_=self.lstm(x)
        return self.head(z[:,-1,:])

class CausalConv(nn.Module):
    def __init__(self,cin,cout,kernel=2,dilation=1):
        super().__init__()
        self.pad=(kernel-1)*dilation
        self.conv=nn.Conv1d(cin,cout,kernel_size=kernel,dilation=dilation)
    def forward(self,x):
        return self.conv(nn.functional.pad(x,(self.pad,0)))

class MultiTCN(nn.Module):
    def __init__(self,channels=32,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(
            CausalConv(21,channels,2,1),nn.ReLU(),nn.Dropout(dropout),
            CausalConv(channels,channels,2,2),nn.ReLU(),nn.Dropout(dropout)
        )
        self.head=nn.Linear(channels,7)
    def forward(self,x):
        z=self.net(x.transpose(1,2))
        return self.head(z[:,:,-1])

CANDIDATES={
 'LSTM':{'LSTM_h16':lambda:MultiLSTM(16),
         'LSTM_h32':lambda:MultiLSTM(32),
         'LSTM_h64':lambda:MultiLSTM(64)},
 'TCN':{'TCN_c16':lambda:MultiTCN(16),
        'TCN_c32':lambda:MultiTCN(32),
        'TCN_c64':lambda:MultiTCN(64)}
}

def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass

def loader(d,split,shuffle=False):
    x=d[split]
    return DataLoader(TensorDataset(torch.from_numpy(x['X']),torch.from_numpy(x['Y'])),
                      batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=0)

@torch.no_grad()
def predict(model,d,split):
    model.eval()
    return np.concatenate([model(x.to(DEVICE)).cpu().numpy()
                           for x,_ in loader(d,split,False)],axis=0)

def train(ctor,d,seed,checkpoint):
    set_seed(seed)  # Before initialization.
    model=ctor().to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    best=float('inf');bad=0;history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train();losses=[]
        for x,y in loader(d,'train',True):
            x,y=x.to(DEVICE),y.to(DEVICE)
            opt.zero_grad()
            loss=nn.functional.mse_loss(model(x),y)
            loss.backward();opt.step()
            losses.append(float(loss.item()))
        valpred=predict(model,d,'validation')
        valrmse=metrics(d['validation']['Y'],valpred)['RMSE']
        history.append({'epoch':epoch,'train_MSE':float(np.mean(losses)),
                        'validation_RMSE':valrmse})
        if valrmse<best-1e-7:
            best=valrmse;bad=0
            torch.save(model.state_dict(),checkpoint)
        else:
            bad+=1
            if bad>=PATIENCE:break
    model.load_state_dict(torch.load(checkpoint,map_location=DEVICE,weights_only=True))
    return model,pd.DataFrame(history),best


In [ ]:
class MultiLSTM(nn.Module):
    def __init__(self,hidden=32):
        super().__init__()
        self.lstm=nn.LSTM(input_size=21,hidden_size=hidden,batch_first=True)
        self.head=nn.Linear(hidden,7)
    def forward(self,x):
        z,_=self.lstm(x)
        return self.head(z[:,-1,:])

class CausalConv(nn.Module):
    def __init__(self,cin,cout,kernel=2,dilation=1):
        super().__init__()
        self.pad=(kernel-1)*dilation
        self.conv=nn.Conv1d(cin,cout,kernel_size=kernel,dilation=dilation)
    def forward(self,x):
        return self.conv(nn.functional.pad(x,(self.pad,0)))

class MultiTCN(nn.Module):
    def __init__(self,channels=32,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(
            CausalConv(21,channels,2,1),nn.ReLU(),nn.Dropout(dropout),
            CausalConv(channels,channels,2,2),nn.ReLU(),nn.Dropout(dropout)
        )
        self.head=nn.Linear(channels,7)
    def forward(self,x):
        z=self.net(x.transpose(1,2))
        return self.head(z[:,:,-1])

CANDIDATES={
 'LSTM':{'LSTM_h16':lambda:MultiLSTM(16),
         'LSTM_h32':lambda:MultiLSTM(32),
         'LSTM_h64':lambda:MultiLSTM(64)},
 'TCN':{'TCN_c16':lambda:MultiTCN(16),
        'TCN_c32':lambda:MultiTCN(32),
        'TCN_c64':lambda:MultiTCN(64)}
}

def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass

def loader(d,split,shuffle=False):
    x=d[split]
    return DataLoader(TensorDataset(torch.from_numpy(x['X']),torch.from_numpy(x['Y'])),
                      batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=0)

@torch.no_grad()
def predict(model,d,split):
    model.eval()
    return np.concatenate([model(x.to(DEVICE)).cpu().numpy()
                           for x,_ in loader(d,split,False)],axis=0)

def train(ctor,d,seed,checkpoint):
    set_seed(seed)  # Before initialization.
    model=ctor().to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    best=float('inf');bad=0;history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train();losses=[]
        for x,y in loader(d,'train',True):
            x,y=x.to(DEVICE),y.to(DEVICE)
            opt.zero_grad()
            loss=nn.functional.mse_loss(model(x),y)
            loss.backward();opt.step()
            losses.append(float(loss.item()))
        valpred=predict(model,d,'validation')
        valrmse=metrics(d['validation']['Y'],valpred)['RMSE']
        history.append({'epoch':epoch,'train_MSE':float(np.mean(losses)),
                        'validation_RMSE':valrmse})
        if valrmse<best-1e-7:
            best=valrmse;bad=0
            torch.save(model.state_dict(),checkpoint)
        else:
            bad+=1
            if bad>=PATIENCE:break
    try:
        weights=torch.load(checkpoint,map_location=DEVICE,weights_only=True)
    except TypeError:
        weights=torch.load(checkpoint,map_location=DEVICE)
    model.load_state_dict(weights)
    return model,pd.DataFrame(history),best


## Exécution : 24 entraînements finaux

La sélection d'architecture est **reprise de E3-E3**. Il n'y a donc **aucun entraînement de sélection supplémentaire**. Le notebook enregistre les historiques et checkpoints dans un nouveau dossier. Si une exécution est interrompue, relancer le notebook recommence les entraînements : les checkpoints ne sont pas des états complets d'optimiseur.


In [ ]:
run_rows=[];prediction_frames=[];per_device_rows=[];configuration_rows=[]
for h in HORIZONS:
    print(f'\\n===== HORIZON {h} MIN =====')
    d,devices=load_horizon(h)
    chosen={family:selected.loc[(h,family)][0] for family in ('LSTM','TCN')}
    assert all(chosen[family] in CANDIDATES[family] for family in chosen)
    print('Fixed architectures from E3-E3:',chosen)
    for family in ('LSTM','TCN'):
        name=chosen[family]
        for seed in SEEDS:
            checkpoint=OUT/f'h{h:03d}_{family}_seed{seed}.pt'
            start=time.perf_counter()
            model,hist,best=train(CANDIDATES[family][name],d,seed,checkpoint)
            hist.to_csv(OUT/f'h{h:03d}_{family}_seed{seed}_history.csv',index=False)
            configuration_rows.append({
                'horizon_min':h,'model':family,'configuration':name,'seed':seed,
                'n_parameters':sum(p.numel() for p in model.parameters()),
                'train_snapshots':len(d['train']['Y']),
                'validation_snapshots':len(d['validation']['Y']),
                'test_snapshots':len(d['test']['Y']),
                'best_validation_RMSE':best,'epochs_run':len(hist),
                'fit_seconds':time.perf_counter()-start
            })
            for split in SPLITS:
                part=d[split]
                pred=predict(model,d,split)
                assert pred.shape==part['Y'].shape and np.isfinite(pred).all()
                y=part['Y'].reshape(-1);p=pred.reshape(-1)
                ref=part['persistence']
                for scope,idx in [('full_horizon_split',np.arange(len(y))),
                                  ('common_keys',part['idx'])]:
                    if scope=='common_keys' and split!='test':continue
                    m=metrics(y[idx],p[idx]);r=metrics(y[idx],ref[idx])
                    run_rows.append({'horizon_min':h,'model':family,
                                     'configuration':name,'seed':seed,
                                     'split':split,'scope':scope,**m,
                                     'Persistence_RMSE':r['RMSE'],
                                     'skill_RMSE':1-m['RMSE']/r['RMSE']})
                if split=='test':
                    idx=part['idx']
                    df=part['meta'].iloc[idx].copy().reset_index(drop=True)
                    df['horizon_min']=h;df['model']=family;df['configuration']=name
                    df['seed']=seed;df['y_true']=y[idx];df['y_pred']=p[idx]
                    df['persistence_pred']=ref[idx];df['error']=p[idx]-y[idx]
                    prediction_frames.append(df)
                    for uuid,g in df.groupby('uuid',sort=True):
                        m=metrics(g.y_true,g.y_pred)
                        r=metrics(g.y_true,g.persistence_pred)
                        per_device_rows.append({'horizon_min':h,'model':family,
                                                'seed':seed,'uuid':uuid,**m,
                                                'Persistence_RMSE':r['RMSE'],
                                                'skill_RMSE':1-m['RMSE']/r['RMSE']})
            print('Final',h,family,seed,'val RMSE',round(best,6))
            del model
            if torch.cuda.is_available():torch.cuda.empty_cache()

runs=pd.DataFrame(run_rows)
preds=pd.concat(prediction_frames,ignore_index=True)
per_device=pd.DataFrame(per_device_rows)
configs=pd.DataFrame(configuration_rows)
primary=runs.query("split=='test' and scope=='common_keys'").copy()
assert len(configs)==24 and len(primary)==24
assert len(preds)==4*2*3*13643
assert primary.n.eq(13643).all()
assert not preds.duplicated(['horizon_min','model','seed','uuid','timestamp']).any()
summary=(primary.groupby(['horizon_min','model'],as_index=False)
         .agg(n_seeds=('seed','nunique'),n=('n','first'),
              RMSE_mean=('RMSE','mean'),RMSE_SD=('RMSE','std'),
              MAE_mean=('MAE','mean'),MAE_SD=('MAE','std'),
              R2_mean=('R2','mean'),R2_SD=('R2','std'),
              Bias_mean=('Bias','mean'),Bias_SD=('Bias','std'),
              skill_RMSE_mean=('skill_RMSE','mean'),
              skill_RMSE_SD=('skill_RMSE','std')))
assert summary.n_seeds.eq(3).all()
primary.to_csv(OUT/'00_PRIMARY_common_TEST_per_seed.csv',index=False)
summary.to_csv(OUT/'00_PRIMARY_common_TEST_mean_sd.csv',index=False)
old_configs.to_csv(OUT/'01_ORIGINAL_E3E3_selected_architectures.csv',index=False)
runs.to_csv(OUT/'02_metrics_all_seeds_all_splits.csv',index=False)
preds.to_csv(OUT/'03_common_test_predictions_all_seeds.csv',index=False)
per_device.to_csv(OUT/'04_common_test_per_device_per_seed.csv',index=False)
configs.to_csv(OUT/'05_selected_configs_and_training.csv',index=False)
protocol={
 'experiment':'E3-E3: C2 multidevice sequence baselines',
 'horizons_minutes':list(HORIZONS),
 'inputs':'same X_graph_scaled [snapshots,5,7,3] as E3-E2, reshaped [snapshots,5,21]',
 'outputs':'seven future temperatures [snapshots,7]',
 'common_test_nodes':13643,'selection':'architectures fixed from E3-E3 validation selection, no new architecture search',
 'final_seeds':list(SEEDS),'training':'full horizon-specific train snapshots',
 'fixed_architectures':{f'h{h:03d}_{family}':selected.loc[(h,family)][0] for h in HORIZONS for family in ('LSTM','TCN')},
 'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'patience':PATIENCE,
 'optimizer':'Adam','learning_rate':LR,'weight_decay':WEIGHT_DECAY,
 'note':'same available information, not equal model capacity',
 'historical_results':'unchanged; E3-E3 configurations read-only','python':platform.python_version(),
 'torch':torch.__version__
}
(OUT/'06_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print(summary.to_string(index=False))
print('Results saved:',OUT)

# Direct numerical comparison, paired by horizon/model/seed (descriptive, not a statistical test).
old=pd.read_csv(PREVIOUS/'00_PRIMARY_common_TEST_per_seed.csv')
compare=primary.merge(old[['horizon_min','model','seed','RMSE','MAE']],
                      on=['horizon_min','model','seed'],suffixes=('_200','_80'),validate='one_to_one')
assert len(compare)==24
compare['RMSE_improvement_percent']=100*(compare.RMSE_80-compare.RMSE_200)/compare.RMSE_80
compare.to_csv(OUT/'07_comparison_80_vs_200_epochs.csv',index=False)
print('\n80 vs 200 epochs (mean RMSE by horizon/model):')
print(compare.groupby(['horizon_min','model'])[['RMSE_80','RMSE_200','RMSE_improvement_percent']].mean().to_string())


## Fichiers à transmettre

- `00_PRIMARY_common_TEST_mean_sd.csv`
- `00_PRIMARY_common_TEST_per_seed.csv`
- `05_selected_configs_and_training.csv`
- `07_comparison_80_vs_200_epochs.csv`
- `06_protocol.json`

Idéalement, transmettre le dossier `outputs/E3_F2B_LSTM_TCN_200epochs/` compressé en ZIP, y compris les historiques. **Ne pas interpréter un gain de test comme critère de sélection** : la décision d'entraînement est fixée à l'avance, les checkpoints sont sélectionnés sur validation.
